In [61]:
import pandas as pd
import yfinance as yf
import numpy as np
import ta
import warnings
warnings.filterwarnings('ignore')

In [62]:
# take 5 Indian stocks
stocks = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS']
data = yf.download(stocks, start='2015-01-01', end='2019-12-31')

[*********************100%***********************]  4 of 4 completed


In [63]:
def add_indicators(data, stocks):
    for stock in stocks:
        # RSI
        data[('RSI', stock)] = ta.momentum.RSIIndicator(data[('Close', stock)], window=14).rsi()
        # MACD
        data[('MACD', stock)] = ta.trend.macd(data[('Close', stock)])
        # EMA
        data[('EMA', stock)] = ta.trend.ema_indicator(data[('Close', stock)], window=14)
        # SMA
        data[('SMA', stock)] = ta.trend.sma_indicator(data[('Close', stock)], window=14)
        # ADXR
        data[('ADXR', stock)] = ta.trend.ADXIndicator(data[('High', stock)], data[('Low', stock)], data[('Close', stock)], window=14).adx()
        # ATR
        data[('ATR', stock)] = ta.volatility.AverageTrueRange(data[('High', stock)], data[('Low', stock)], data[('Close', stock)], window=14).average_true_range()


In [8]:
# drop NaN values
add_indicators(data, stocks)
data.dropna(inplace=True)
data.head(2)

Price                       Adj Close                                       \
Ticker                    HDFCBANK.NS     INFY.NS RELIANCE.NS       TCS.NS   
Date                                                                         
2015-03-17 00:00:00+00:00  489.370697  434.687378  176.924194  1087.951538   
2015-03-18 00:00:00+00:00  491.914612  432.553192  179.360718  1077.697998   

Price                           Close                                    \
Ticker                    HDFCBANK.NS  INFY.NS RELIANCE.NS       TCS.NS   
Date                                                                      
2015-03-17 00:00:00+00:00  529.025024  560.125  195.861496  1291.849976   
2015-03-18 00:00:00+00:00  531.775024  557.375  198.558777  1279.675049   

Price                            High              ...         EMA  \
Ticker                    HDFCBANK.NS     INFY.NS  ... HDFCBANK.NS   
Date                                               ...               
2015-03-17 00:00:00+00:00  532.950012  569.875000  ...  529.329585   
2015-03-18 00:00:00+00:00  535.950012  565.712524  ...  529.655644   

Price                             SMA        ADXR         ATR        RSI  \
Ticker                    HDFCBANK.NS HDFCBANK.NS HDFCBANK.NS    INFY.NS   
Date                                                                       
2015-03-17 00:00:00+00:00  529.848219         0.0   11.477732  48.561444   
2015-03-18 00:00:00+00:00  530.244651         0.0   11.325754  46.716639   

Price                          MACD         EMA         SMA    ADXR        ATR  
Ticker                      INFY.NS     INFY.NS     INFY.NS INFY.NS    INFY.NS  
Date                                                                            
2015-03-17 00:00:00+00:00 -2.012714  560.589661  561.952680     0.0  13.426914  
2015-03-18 00:00:00+00:00 -2.079196  560.161040  560.379464     0.0  13.187493  

[2 rows x 48 columns]

In [9]:
features = ['RSI', 'MACD', 'EMA', 'SMA', 'ADXR', 'ATR']
data.xs("RELIANCE.NS", axis=1, level=1)[features][:2]

Price,RSI,MACD,EMA,SMA,ADXR,ATR
Date,,,,,,
2015-03-17 00:00:00+00:00,43.189259,-2.547972,197.456918,197.171767,0.0,5.005039
2015-03-18 00:00:00+00:00,48.097151,-2.238762,197.603833,197.588115,0.0,4.934082


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

# Function to load and preprocess the data
def load_and_preprocess_data(filepath, feature_column='Close'):
    """
    Load the dataset and extract the relevant feature (e.g., 'Close' prices).
    """
    data = pd.read_csv(filepath)
    prices = data[feature_column].values.reshape(-1, 1)
    return prices

# Function to split and scale the data
def split_and_scale_data(data, test_size=0.2):
    """
    Split the data into training and testing sets, and scale them.
    """
    # Split the data
    train_data, test_data = train_test_split(data, test_size=test_size, shuffle=False)
    
    # Scale the data
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_train_data = scaler.fit_transform(train_data)
    scaled_test_data = scaler.transform(test_data)
    
    return scaled_train_data, scaled_test_data, scaler

# Function to create sequences for LSTM
def create_sequences(data, seq_length):
    """
    Create input sequences and corresponding target values for the LSTM model.
    """
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

# Function to build the LSTM model
def build_lstm_model(input_shape):
    """
    Build and compile the LSTM model.
    """
    model = Sequential()
    
    # First LSTM layer
    model.add(LSTM(units=50, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))
    
    # Second LSTM layer
    model.add(LSTM(units=50, return_sequences=False))
    model.add(Dropout(0.2))
    
    # Output layer
    model.add(Dense(units=1))
    
    # Compile the model
    model.compile(optimizer='adam', loss='mean_squared_error')
    
    return model

# Function to train the model
def train_model(model, X_train, y_train, epochs=50, batch_size=32):
    """
    Train the LSTM model.
    """
    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=1)
    return model

# Function to make predictions
def make_predictions(model, X_test, scaler):
    """
    Make predictions using the trained model and inverse transform the scaled values.
    """
    predicted_prices = model.predict(X_test)
    predicted_prices = scaler.inverse_transform(predicted_prices)
    return predicted_prices

# Function to evaluate the model
def evaluate_model(actual_prices, predicted_prices):
    """
    Evaluate the model using metrics like RMSE.
    """
    from sklearn.metrics import mean_squared_error
    mse = mean_squared_error(actual_prices, predicted_prices)
    rmse = np.sqrt(mse)
    print(f"RMSE: {rmse}")
    return rmse

# Function to visualize the results
def plot_results(actual_prices, predicted_prices):
    """
    Plot the actual vs predicted stock prices.
    """
    plt.figure(figsize=(14, 5))
    plt.plot(actual_prices, color='blue', label='Actual Stock Price')
    plt.plot(predicted_prices, color='red', label='Predicted Stock Price')
    plt.title('Stock Price Prediction')
    plt.xlabel('Time')
    plt.ylabel('Stock Price')
    plt.legend()
    plt.show()


In [ ]:
prices = load_and_preprocess_data(filepath)
    
    # Step 2: Split and scale the data
scaled_train_data, scaled_test_data, scaler = split_and_scale_data(prices)
    
    # Step 3: Create sequences for LSTM
seq_length = 60
X_train, y_train = create_sequences(scaled_train_data, seq_length)
X_test, y_test = create_sequences(scaled_test_data, seq_length)
    
    # Reshape X for LSTM input
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)    
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
    
    # Step 4: Build the LSTM model
input_shape = (X_train.shape[1], 1)
model = build_lstm_model(input_shape)
    
    # Step 5: Train the model
model = train_model(model, X_train, y_train)
    
    # Step 6: Make predictions
predicted_prices = make_predictions(model, X_test, scaler)
    
    # Inverse transform the actual prices
actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))
    
    # Step 7: Evaluate the model
evaluate_model(actual_prices, predicted_prices)
    
    # Step 8: Visualize the results
plot_results(actual_prices, predicted_prices)


In [9]:
def sod(num):
    ans = 0
    while(num != 0):
        rem = num % 10
        ans += rem
        num = (num-rem)/10
    return ans

sod(1034302000)

13.0